# Week 3 – Feast Feature Store for the IRIS pipeline
Roll no. 21f2000602 · Offline store: local Parquet (Task 6: BigQuery) · Online store: SQLite

In [1]:
%pip install "feast[gcp]" scikit-learn pandas joblib dvc dvc-gs

Note: you may need to restart the kernel to use updated packages.


In [2]:
%pip install --upgrade "scikit-learn>=1.5" scipy

Note: you may need to restart the kernel to use updated packages.


In [3]:
import numpy, sklearn, scipy
print("numpy", numpy.__version__, "| scikit-learn", sklearn.__version__, "| scipy", scipy.__version__)

numpy 2.5.4 | scikit-learn 1.9.1 | scipy 1.18.1


 - Checking Feast Version

In [4]:
!feast version

Feast SDK Version: "0.67.0"


### Initializing Feast

In [5]:
!feast init iris_feature_store


Creating a new Feast repository in /home/jupyter/21f2000602_MLOPS_WEEKLY_ASSIGNMENT/iris_feature_store.



In [6]:
!find iris_feature_store -type f -not -path "*/__pycache__/*"

iris_feature_store/README.md
iris_feature_store/feature_repo/feature_store.yaml
iris_feature_store/feature_repo/feature_definitions.py
iris_feature_store/feature_repo/test_workflow.py
iris_feature_store/feature_repo/__init__.py
iris_feature_store/feature_repo/data/driver_stats.parquet
iris_feature_store/feature_repo/data/driver_quality_labels.parquet
iris_feature_store/__init__.py
iris_feature_store/.gitignore


 - Checking Config file

In [7]:
!cat iris_feature_store/feature_repo/feature_store.yaml

project: iris_feature_store
# By default, the registry is a file (but can be turned into a more scalable SQL-backed registry)
registry: data/registry.db
# The provider primarily specifies default offline / online stores & storing the registry in a given cloud
provider: local
online_store:
    type: sqlite
    path: data/online_store.db
entity_key_serialization_version: 3
# By default, no_auth for authentication and authorization, other possible values kubernetes and oidc. Refer the documentation for more details.
auth:
    type: no_auth


- Explicitly write the oofline store config for later use

In [8]:
%%writefile -a iris_feature_store/feature_repo/feature_store.yaml
offline_store:
    type: file                   # offline (historical) store = local Parquet files via pandas

Appending to iris_feature_store/feature_repo/feature_store.yaml


In [9]:
!cat iris_feature_store/feature_repo/feature_store.yaml

project: iris_feature_store
# By default, the registry is a file (but can be turned into a more scalable SQL-backed registry)
registry: data/registry.db
# The provider primarily specifies default offline / online stores & storing the registry in a given cloud
provider: local
online_store:
    type: sqlite
    path: data/online_store.db
entity_key_serialization_version: 3
# By default, no_auth for authentication and authorization, other possible values kubernetes and oidc. Refer the documentation for more details.
auth:
    type: no_auth
offline_store:
    type: file                   # offline (historical) store = local Parquet files via pandas


#### (Optional) Removing Unnecessary files

In [10]:
!cd iris_feature_store/feature_repo && rm -f feature_definitions.py example_repo.py test_workflow.py data/driver_*.parquet
!rm -f iris_feature_store/README.md
!find iris_feature_store -type f -not -path "*/__pycache__/*"

iris_feature_store/feature_repo/feature_store.yaml
iris_feature_store/feature_repo/__init__.py
iris_feature_store/__init__.py
iris_feature_store/.gitignore


#### Keep Generated files out of git

In [11]:
%%writefile iris_feature_store/feature_repo/.gitignore
# Generated by Feast - rebuilt by `feast apply` / `feast materialize`, never committed
data/registry.db
data/online_store.db
# Parquet copy of the dataset - regenerated by the notebook from the DVC-tracked CSV
data/*.parquet

Writing iris_feature_store/feature_repo/.gitignore


### Downloading and Versioning Data

In [12]:
!dvc import https://github.com/IITMBSMLOps/ga_resources.git iris_data_adapted_for_feast.csv \
    -o iris_feature_store/feature_repo/data/iris_data_adapted_for_feast.csv --rev week_3

Importing 'iris_data_adapted_for_feast.csv (https://github.com/IITMBSMLOps/ga_resources.git)' -> 'iris_feature_store/feature_repo/data/iris_data_adapted_for_feast.csv'
                                                                               
To track the changes with git, run:

	git add iris_feature_store/feature_repo/data/iris_data_adapted_for_feast.csv.dvc iris_feature_store/feature_repo/data/.gitignore

To enable auto staging, run:

	dvc config core.autostage true


- Data Exploration

In [13]:
import pandas as pd

csv_path = "iris_feature_store/feature_repo/data/iris_data_adapted_for_feast.csv"
raw = pd.read_csv(csv_path, parse_dates=["event_timestamp", "created_timestamp"])

print(raw.shape)                                     # rows, columns
print(raw.groupby("iris_id")["species"].agg(["count", "first"]))   # rows + species per plant
print(raw["event_timestamp"].min(), "→", raw["event_timestamp"].max())
raw.head()

(45, 8)
         count       first
iris_id                   
1001        15  versicolor
1002        15      setosa
1003        15      setosa
2025-09-17 10:40:17.102131 → 2025-10-01 10:40:17.102131


,event_timestamp,iris_id,sepal_length,sepal_width,petal_length,petal_width,species,created_timestamp
0,2025-09-17 10:40:17.102131,1001,5.52,2.53,3.86,1.13,versicolor,2025-10-02 10:40:17.172178
1,2025-09-18 10:40:17.102131,1001,5.50,2.24,3.60,1.08,versicolor,2025-10-02 10:40:17.172178
2,2025-09-19 10:40:17.102131,1001,5.55,2.47,3.75,1.08,versicolor,2025-10-02 10:40:17.172178
3,2025-09-20 10:40:17.102131,1001,5.45,2.37,3.92,1.20,versicolor,2025-10-02 10:40:17.172178
4,2025-09-21 10:40:17.102131,1001,5.65,2.52,3.95,1.17,versicolor,2025-10-02 10:40:17.172178


- Convert to parquet

In [14]:
for col in ["event_timestamp", "created_timestamp"]:
    raw[col] = raw[col].dt.tz_localize("UTC")        # mark times as UTC

parquet_path = "iris_feature_store/feature_repo/data/iris_data.parquet"
raw.to_parquet(parquet_path, index=False)
print(pd.read_parquet(parquet_path).dtypes)

event_timestamp      datetime64[ns, UTC]
iris_id                            int64
sepal_length                     float64
sepal_width                      float64
petal_length                     float64
petal_width                      float64
species                           object
created_timestamp    datetime64[ns, UTC]
dtype: object


### Writing Features for the data

In [15]:
%%writefile iris_feature_store/feature_repo/iris_features.py
from datetime import timedelta

from feast import Entity, FeatureView, Field, FileSource
from feast.types import Float64
from feast.value_type import ValueType

# ENTITY: the "thing" features describe. join_keys = the column that identifies it.
iris = Entity(
    name="iris",
    join_keys=["iris_id"],
    value_type=ValueType.INT64,
    description="One iris plant, tracked over time",
)

# DATA SOURCE: where raw feature values live + which column says WHEN each was measured.
iris_source = FileSource(
    name="iris_source",
    path="data/iris_data.parquet",                 # relative to this feature repo
    timestamp_field="event_timestamp",             # when the measurement happened
    created_timestamp_column="created_timestamp",  # when the row was written (tie-breaker)
)

# FEATURE VIEW: which columns are features, for which entity, from which source.
iris_features = FeatureView(
    name="iris_features",
    entities=[iris],
    ttl=timedelta(days=365 * 5),                   # how old a value may be and still count
    schema=[
        Field(name="sepal_length", dtype=Float64),
        Field(name="sepal_width",  dtype=Float64),
        Field(name="petal_length", dtype=Float64),
        Field(name="petal_width",  dtype=Float64),
    ],
    online=True,                                   # allow materializing into the online store
    source=iris_source,
)

Writing iris_feature_store/feature_repo/iris_features.py


- CHecking File write

In [16]:
!cat iris_feature_store/feature_repo/iris_features.py | head -15
!ls -la iris_feature_store/feature_repo/data

from datetime import timedelta

from feast import Entity, FeatureView, Field, FileSource
from feast.types import Float64
from feast.value_type import ValueType

# ENTITY: the "thing" features describe. join_keys = the column that identifies it.
iris = Entity(
    name="iris",
    join_keys=["iris_id"],
    value_type=ValueType.INT64,
    description="One iris plant, tracked over time",
)

# DATA SOURCE: where raw feature values live + which column says WHEN each was measured.
total 28
drwxr-xr-x 2 jupyter jupyter 4096 Oct 25 19:56 .
drwxr-xr-x 4 jupyter jupyter 4096 Oct 25 19:58 ..
-rw-r--r-- 1 jupyter jupyter   33 Oct 25 19:55 .gitignore
-rw-r--r-- 1 jupyter jupyter 6670 Oct 25 19:56 iris_data.parquet
-rw-r--r-- 1 jupyter jupyter 4009 Oct 25 19:55 iris_data_adapted_for_feast.csv
-rw-r--r-- 1 jupyter jupyter  344 Oct 25 19:55 iris_data_adapted_for_feast.csv.dvc


### Apply definitions & materialize into the online store

In [17]:
%%bash
cd iris_feature_store/feature_repo
feast apply

No project found in the repository. Using project name iris_feature_store defined in feature_store.yaml
Applying changes for project iris_feature_store
Created project iris_feature_store
Created entity iris
Created feature view iris_features

Created sqlite table iris_feature_store_iris_features



- Verify Changes

In [18]:
%%bash
cd iris_feature_store/feature_repo
feast entities list
feast feature-views list

NAME    DESCRIPTION                        TYPE
iris    One iris plant, tracked over time  ValueType.INT64
NAME           ENTITIES    TYPE         ENABLED    STATE
iris_features  {'iris'}    FeatureView  Yes        STATE_UNSPECIFIED


- Check by looking inside the store

In [19]:
import sqlite3

db = "iris_feature_store/feature_repo/data/online_store.db"
table = "iris_feature_store_iris_features"

con = sqlite3.connect(db)
print("Tables:", con.execute("SELECT name FROM sqlite_master WHERE type='table'").fetchall())
print("Rows before materialize:", con.execute(f"SELECT COUNT(*) FROM {table}").fetchone()[0])
con.close()

Tables: [('iris_feature_store_iris_features',)]
Rows before materialize: 0


#### Materialize

In [20]:
%%bash
cd iris_feature_store/feature_repo
feast materialize 2025-09-01T00:00:00 2025-10-02T00:00:00

Materializing 1 feature views from 2025-09-01 00:00:00+00:00 to 2025-10-02 00:00:00+00:00 into the sqlite online store.

iris_features:


- verify Materialization

In [21]:
from datetime import datetime, timezone

con = sqlite3.connect(db)
n = con.execute(f"SELECT COUNT(*) FROM {table}").fetchone()[0]
print("Rows after materialize:", n)

for feat, ts in con.execute(f"SELECT feature_name, event_ts FROM {table} LIMIT 4"):
    print(f"{feat:13s} event_ts = {datetime.fromtimestamp(ts, tz=timezone.utc)}")
con.close()

Rows after materialize: 12
sepal_length  event_ts = 2025-10-01 10:40:17+00:00
sepal_width   event_ts = 2025-10-01 10:40:17+00:00
petal_length  event_ts = 2025-10-01 10:40:17+00:00
petal_width   event_ts = 2025-10-01 10:40:17+00:00


In [22]:
!cd iris_feature_store/feature_repo && feast feature-views list

NAME           ENTITIES    TYPE         ENABLED    STATE
iris_features  {'iris'}    FeatureView  Yes        AVAILABLE_ONLINE


### Fetch training features from the OFFLINE store and train

In [23]:
from feast import FeatureStore

fs = FeatureStore(repo_path="iris_feature_store/feature_repo")

FEATURES = [
    "iris_features:sepal_length",
    "iris_features:sepal_width",
    "iris_features:petal_length",
    "iris_features:petal_width",
]
FEATURE_COLS = [f.split(":")[1] for f in FEATURES]
print(FEATURE_COLS)

['sepal_length', 'sepal_width', 'petal_length', 'petal_width']


In [24]:
labels = pd.read_parquet("iris_feature_store/feature_repo/data/iris_data.parquet")
entity_df = labels[["iris_id", "event_timestamp", "species"]]

print(entity_df.shape)
entity_df.head()

(45, 3)


,iris_id,event_timestamp,species
0,1001,2025-09-17 10:40:17.102131+00:00,versicolor
1,1001,2025-09-18 10:40:17.102131+00:00,versicolor
2,1001,2025-09-19 10:40:17.102131+00:00,versicolor
3,1001,2025-09-20 10:40:17.102131+00:00,versicolor
4,1001,2025-09-21 10:40:17.102131+00:00,versicolor


In [25]:
training_df = fs.get_historical_features(
    entity_df=entity_df,
    features=FEATURES,
).to_df()

print(training_df.shape)
training_df.head()

(45, 7)


,iris_id,event_timestamp,species,sepal_length,sepal_width,petal_length,petal_width
0,1001,2025-09-17 10:40:17.102131+00:00,versicolor,5.52,2.53,3.86,1.13
1,1003,2025-09-17 10:40:17.102131+00:00,setosa,5.09,3.42,1.34,0.22
2,1002,2025-09-17 10:40:17.102131+00:00,setosa,4.92,3.05,1.43,0.29
3,1001,2025-09-18 10:40:17.102131+00:00,versicolor,5.50,2.24,3.60,1.08
4,1002,2025-09-18 10:40:17.102131+00:00,setosa,5.05,2.94,1.53,0.31


In [26]:
## Verificaiton

check = training_df.merge(labels, on=["iris_id", "event_timestamp"], suffixes=("_feast", "_raw"))
match = all((check[c + "_feast"] == check[c + "_raw"]).all() for c in FEATURE_COLS)
print("Rows compared:", len(check), "| Offline features identical to raw data:", match)

Rows compared: 45 | Offline features identical to raw data: True


 - Point-in-time Join

In [27]:
demo = pd.DataFrame({
    "iris_id": [1001, 1001, 1001, 9999],
    "event_timestamp": pd.to_datetime([
        "2025-09-20 18:00",   # evening of Sep 20                → Sep 20 morning's values
        "2025-09-21 09:00",   # before Sep 21's 10:40 measurement → still Sep 20's values
        "2025-09-10 09:00",   # before plant 1001's first data    → no value exists → row dropped
        "2025-09-20 18:00",   # plant that doesn't exist          → kept, with NaN features
    ]).tz_localize("UTC"),
})
pit = fs.get_historical_features(entity_df=demo, features=FEATURES).to_df()

print(f"Rows requested: {len(demo)} | Rows returned: {len(pit)}")
pit.sort_values(["iris_id", "event_timestamp"])

Rows requested: 4 | Rows returned: 3


,iris_id,event_timestamp,sepal_length,sepal_width,petal_length,petal_width
1,1001,2025-09-20 18:00:00+00:00,5.45,2.37,3.92,1.2
2,1001,2025-09-21 09:00:00+00:00,5.45,2.37,3.92,1.2
0,9999,2025-09-20 18:00:00+00:00,NaN,NaN,NaN,NaN


In [28]:
labels[(labels.iris_id == 1001) & (labels.event_timestamp.dt.day.isin([20, 21]))][["event_timestamp"] + FEATURE_COLS]

,event_timestamp,sepal_length,sepal_width,petal_length,petal_width
3,2025-09-20 10:40:17.102131+00:00,5.45,2.37,3.92,1.20
4,2025-09-21 10:40:17.102131+00:00,5.65,2.52,3.95,1.17


#### Training the Model

In [29]:
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score

X = training_df[FEATURE_COLS]                 # inputs: ONLY the 4 features from Feast
y = training_df["species"]                    # label: passed through the entity dataframe

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.4, stratify=y, random_state=42
)

model = DecisionTreeClassifier(max_depth=3, random_state=1)
model.fit(X_train, y_train)
acc = accuracy_score(y_test, model.predict(X_test))
print(f"Train rows: {len(X_train)} | Test rows: {len(X_test)} | Test accuracy: {acc:.3f}")

Train rows: 27 | Test rows: 18 | Test accuracy: 1.000


In [30]:
import os, json, joblib

os.makedirs("artifacts", exist_ok=True)
joblib.dump(model, "artifacts/iris_feast_model.joblib")

metrics = {
    "training_rows_from_offline_store": len(training_df),
    "test_accuracy": round(acc, 4),
    "offline_features_match_raw": bool(match),
}
with open("week3_metrics.json", "w") as f:
    json.dump(metrics, f, indent=2)
print(metrics)

{'training_rows_from_offline_store': 45, 'test_accuracy': 1.0, 'offline_features_match_raw': True}


### Real-time inference from the ONLINE store

#### Prediction Service

In [31]:
import joblib
from feast import FeatureStore

class IrisPredictor:
    """Simulates a real-time prediction service: input = iris IDs, output = predicted species."""

    def __init__(self, model_path, repo_path):
        self.model = joblib.load(model_path)              # load the trained model once, at startup
        self.fs = FeatureStore(repo_path=repo_path)       # connect to the same feature store

    def get_features(self, iris_ids):
        response = self.fs.get_online_features(
            features=FEATURES,                            # SAME feature list used in training
            entity_rows=[{"iris_id": i} for i in iris_ids],
        )
        return response.to_df()

    def predict(self, iris_ids):
        feats = self.get_features(iris_ids)
        feats["predicted_species"] = self.model.predict(feats[FEATURE_COLS])   # select by NAME
        return feats

predictor = IrisPredictor("artifacts/iris_feast_model.joblib", "iris_feature_store/feature_repo")

##### Live Prediction

In [32]:
import time

request_ids = [1001, 1002, 1003]

t0 = time.perf_counter()
online_result = predictor.predict(request_ids)
latency_ms = (time.perf_counter() - t0) * 1000

print(f"Online lookup + prediction for {len(request_ids)} plants took {latency_ms:.1f} ms")
online_result

Online lookup + prediction for 3 plants took 33.4 ms


,iris_id,sepal_width,petal_width,petal_length,sepal_length,predicted_species
0,1001,2.36,1.09,3.84,5.45,versicolor
1,1002,2.90,0.20,1.29,4.84,setosa
2,1003,3.40,0.29,1.19,4.85,setosa


##### Consistency Check

In [33]:
raw_data = pd.read_parquet("iris_feature_store/feature_repo/data/iris_data.parquet")

# "Raw path": skip Feast entirely - take each plant's most recent row straight from the file
latest_raw = (raw_data.sort_values("event_timestamp")
                      .groupby("iris_id").tail(1)
                      .set_index("iris_id")
                      .loc[request_ids])

online_feats = online_result.set_index("iris_id").loc[request_ids]

features_match = (online_feats[FEATURE_COLS].values == latest_raw[FEATURE_COLS].values).all()
raw_preds = predictor.model.predict(latest_raw[FEATURE_COLS])
preds_match = (online_feats["predicted_species"].values == raw_preds).all()

comparison = pd.DataFrame({
    "online_prediction": online_feats["predicted_species"].values,
    "raw_data_prediction": raw_preds,
    "true_species": latest_raw["species"].values,
    "latest_measurement_at": latest_raw["event_timestamp"].values,
}, index=request_ids)

print("Online features identical to latest raw rows:", features_match)
print("Online predictions identical to raw-data predictions:", preds_match)
comparison

Online features identical to latest raw rows: True
Online predictions identical to raw-data predictions: True


,online_prediction,raw_data_prediction,true_species,latest_measurement_at
1001,versicolor,versicolor,versicolor,2025-10-01 10:40:17.102131
1002,setosa,setosa,setosa,2025-10-01 10:40:17.102131
1003,setosa,setosa,setosa,2025-10-01 10:40:17.102131


- CHecking Unknown Plant

In [34]:
predictor.get_features([1001, 9999])

,iris_id,sepal_width,petal_width,petal_length,sepal_length
0,1001,2.36,1.09,3.84,5.45
1,9999,NaN,NaN,NaN,NaN


- Record Keeping

In [35]:
metrics.update({
    "online_request_ids": request_ids,
    "online_predictions": online_feats["predicted_species"].tolist(),
    "online_features_match_latest_raw": bool(features_match),
    "online_predictions_match_raw_predictions": bool(preds_match),
    "online_latency_ms": round(latency_ms, 1),
})
with open("week3_metrics.json", "w") as f:
    json.dump(metrics, f, indent=2)
print(json.dumps(metrics, indent=2))

{
  "training_rows_from_offline_store": 45,
  "test_accuracy": 1.0,
  "offline_features_match_raw": true,
  "online_request_ids": [
    1001,
    1002,
    1003
  ],
  "online_predictions": [
    "versicolor",
    "setosa",
    "setosa"
  ],
  "online_features_match_latest_raw": true,
  "online_predictions_match_raw_predictions": true,
  "online_latency_ms": 33.4
}


### BigQuery as the offline store

In [36]:
%pip install -q db-dtypes

/opt/micromamba/lib/python3.12/pty.py:95: DeprecationWarning: This process (pid=51568) is multi-threaded, use of forkpty() may lead to deadlocks in the child.
  pid, fd = os.forkpty()


Note: you may need to restart the kernel to use updated packages.


##### Creating A BigQuery Dataset and loading the Iris data

In [37]:
from google.cloud import bigquery

PROJECT_ID = "arcane-sentinel-252018"
BQ_LOCATION = "us-central1"
BQ_DATASET = "feast_iris"
BQ_TABLE = f"{PROJECT_ID}.{BQ_DATASET}.iris_source"

client = bigquery.Client(project=PROJECT_ID, location=BQ_LOCATION)

dataset = bigquery.Dataset(f"{PROJECT_ID}.{BQ_DATASET}")
dataset.location = BQ_LOCATION
client.create_dataset(dataset, exists_ok=True)

raw_data = pd.read_parquet("iris_feature_store/feature_repo/data/iris_data.parquet")
job = client.load_table_from_dataframe(
    raw_data, BQ_TABLE,
    job_config=bigquery.LoadJobConfig(write_disposition="WRITE_TRUNCATE"),
)
job.result()
print("Loaded", client.get_table(BQ_TABLE).num_rows, "rows into", BQ_TABLE)

Loaded 45 rows into arcane-sentinel-252018.feast_iris.iris_source


- CHecking with SQl

In [38]:
client.query(f"""
    SELECT iris_id, species, COUNT(*) AS n_rows,
           MIN(event_timestamp) AS first_measurement, MAX(event_timestamp) AS last_measurement
    FROM `{BQ_TABLE}`
    GROUP BY iris_id, species
    ORDER BY iris_id
""").to_dataframe()

,iris_id,species,n_rows,first_measurement,last_measurement
0,1001,versicolor,15,2025-09-17 10:40:17.102131+00:00,2025-10-01 10:40:17.102131+00:00
1,1002,setosa,15,2025-09-17 10:40:17.102131+00:00,2025-10-01 10:40:17.102131+00:00
2,1003,setosa,15,2025-09-17 10:40:17.102131+00:00,2025-10-01 10:40:17.102131+00:00


##### BigQuery Feature Repo

In [39]:
!mkdir -p iris_feature_store/feature_repo_bq/data

/opt/micromamba/lib/python3.12/pty.py:95: DeprecationWarning: This process (pid=51568) is multi-threaded, use of forkpty() may lead to deadlocks in the child.
  pid, fd = os.forkpty()


In [40]:
%%writefile iris_feature_store/feature_repo_bq/feature_store.yaml
project: iris_feature_store_bq
registry: data/registry.db
provider: gcp
# OFFLINE store moved to BigQuery (was: type: file)
offline_store:
    type: bigquery
    project_id: arcane-sentinel-252018
    dataset: feast_iris            # Feast writes temporary tables here (e.g. the uploaded entity_df)
    location: us-central1
# ONLINE store unchanged: SQLite for low-latency lookups
online_store:
    type: sqlite
    path: data/online_store.db
entity_key_serialization_version: 3
auth:
    type: no_auth

Writing iris_feature_store/feature_repo_bq/feature_store.yaml


##### BQ Feature Definitions

In [41]:
%%writefile iris_feature_store/feature_repo_bq/iris_features_bq.py
from datetime import timedelta

from feast import BigQuerySource, Entity, FeatureView, Field
from feast.types import Float64
from feast.value_type import ValueType

iris = Entity(name="iris", join_keys=["iris_id"], value_type=ValueType.INT64,
              description="One iris plant, tracked over time")

# The ONLY real change vs feature_repo/iris_features.py: FileSource -> BigQuerySource
iris_source = BigQuerySource(
    name="iris_source_bq",
    table="arcane-sentinel-252018.feast_iris.iris_source",
    timestamp_field="event_timestamp",
    created_timestamp_column="created_timestamp",
)

iris_features = FeatureView(
    name="iris_features",
    entities=[iris],
    ttl=timedelta(days=365 * 5),
    schema=[
        Field(name="sepal_length", dtype=Float64),
        Field(name="sepal_width",  dtype=Float64),
        Field(name="petal_length", dtype=Float64),
        Field(name="petal_width",  dtype=Float64),
    ],
    online=True,
    source=iris_source,
)

Writing iris_feature_store/feature_repo_bq/iris_features_bq.py


In [42]:
%%writefile iris_feature_store/feature_repo_bq/.gitignore
data/registry.db
data/online_store.db

Writing iris_feature_store/feature_repo_bq/.gitignore


##### Apply and Materialize

In [ ]:
%%bash
cd iris_feature_store/feature_repo_bq
feast apply
feast materialize 2025-09-01T00:00:00 2025-10-02T00:00:00
feast feature-views list

No project found in the repository. Using project name iris_feature_store_bq defined in feature_store.yaml
Applying changes for project iris_feature_store_bq
Deploying infrastructure for iris_features
Materializing 1 feature views from 2025-09-01 00:00:00+00:00 to 2025-10-02 00:00:00+00:00 into the sqlite online store.

iris_features:


##### Training Code

In [ ]:
fs_bq = FeatureStore(repo_path="iris_feature_store/feature_repo_bq")

t0 = time.perf_counter()
training_df_bq = fs_bq.get_historical_features(
    entity_df=entity_df,          # SAME entity dataframe as Task 4
    features=FEATURES,            # SAME feature list
).to_df()
bq_seconds = time.perf_counter() - t0

t0 = time.perf_counter()
fs.get_historical_features(entity_df=entity_df, features=FEATURES).to_df()
file_seconds = time.perf_counter() - t0

a = training_df.sort_values(["iris_id", "event_timestamp"]).reset_index(drop=True)
b = training_df_bq.sort_values(["iris_id", "event_timestamp"]).reset_index(drop=True)
same_features = (a[FEATURE_COLS].values == b[FEATURE_COLS].values).all()

print(f"Rows from BigQuery: {len(training_df_bq)} | identical to file-store features: {same_features}")
print(f"Offline retrieval time  ->  BigQuery: {bq_seconds:.2f}s   |   local file: {file_seconds:.2f}s")

In [42]:
Xb = training_df_bq[FEATURE_COLS]
yb = training_df_bq["species"]
Xb_tr, Xb_te, yb_tr, yb_te = train_test_split(Xb, yb, test_size=0.4, stratify=yb, random_state=42)

model_bq = DecisionTreeClassifier(max_depth=3, random_state=1).fit(Xb_tr, yb_tr)
acc_bq = accuracy_score(yb_te, model_bq.predict(Xb_te))

online_bq = fs_bq.get_online_features(
    features=FEATURES, entity_rows=[{"iris_id": i} for i in request_ids]
).to_df().set_index("iris_id").loc[request_ids]

preds_bq = model_bq.predict(online_bq[FEATURE_COLS])
print(f"Accuracy (BigQuery-trained): {acc_bq:.3f}  vs  file-trained: {acc:.3f}")
print("Online predictions (BigQuery repo):", list(preds_bq))
print("Same as Task 5 predictions:", list(preds_bq) == online_feats["predicted_species"].tolist())

Accuracy (BigQuery-trained): 1.000  vs  file-trained: 1.000
Online predictions (BigQuery repo): ['versicolor', 'setosa', 'setosa']
Same as Task 5 predictions: True


- Point-in-time case

In [43]:
pit_bq = fs_bq.get_historical_features(entity_df=demo, features=FEATURES).to_df()
print(f"BigQuery – rows requested: {len(demo)} | returned: {len(pit_bq)}")
pit_bq.sort_values(["iris_id", "event_timestamp"])

BigQuery – rows requested: 4 | returned: 4


,iris_id,event_timestamp,sepal_length,sepal_width,petal_length,petal_width
1,1001,2025-09-10 09:00:00+00:00,NaN,NaN,NaN,NaN
2,1001,2025-09-20 18:00:00+00:00,5.45,2.37,3.92,1.2
0,1001,2025-09-21 09:00:00+00:00,5.45,2.37,3.92,1.2
3,9999,2025-09-20 18:00:00+00:00,NaN,NaN,NaN,NaN


##### Configuration changes
- New repo `feature_repo_bq/`: `provider: gcp`, `offline_store: {type: bigquery, project_id, dataset: feast_iris, location: us-central1}`; online store still SQLite.
- Data source: `FileSource(path=parquet)` → `BigQuerySource(table=arcane-sentinel-252018.feast_iris.iris_source)`.
- Feature view, feature list, training and inference code: **unchanged**.

##### Trade-offs observed
| | Local file (Parquet) | BigQuery |
|---|---|---|
| Setup | None | Enable API, create dataset, load table, IAM permissions |
| Offline retrieval time (45 rows) | 0.14 s | 8.68 s (network + job overhead) |
| Scale | Limited by one VM's memory | Petabyte-scale, computed in the warehouse |
| Sharing | Only this VM | Whole team / any service with IAM access |
| Cost | Free | Billed per query & storage (here: within free tier) |
| Point-in-time join | pandas in VM memory | SQL in BigQuery |
| Row with no prior data | Dropped | Valid row with NaN values |
| Best for | Learning, small data, local dev | Production training data |

In [44]:
metrics.update({
    "bigquery_offline_rows": len(training_df_bq),
    "bigquery_features_match_file_store": bool(same_features),
    "bigquery_retrieval_seconds": round(bq_seconds, 2),
    "file_retrieval_seconds": round(file_seconds, 2),
    "bigquery_trained_accuracy": round(acc_bq, 4),
})
with open("week3_metrics.json", "w") as f:
    json.dump(metrics, f, indent=2)